###Settings

In [0]:

# Adds a text box called catalog at the top of the notebook, defaulting to retail_dev. In Section 6 the job passes catalog
dbutils.widgets.text("catalog", "retail_dev")

# Reads the value in that box
CATALOG = dbutils.widgets.get("catalog")

# Every table name without a catalog, such as bronze.erp_inventory_cdc, now means retail_dev.bronze.
# Without this line, your tables would land in the wrong catalog.
spark.sql(f"USE CATALOG retail_dev")

# JDBC_URL is the address of your Azure SQL database
# loginTimeout=60 waits up to a minute while a paused database wakes up.
JDBC_URL = ("jdbc:sqlserver://sql-retail-server-nde.database.windows.net:1433;database=sqldb-erp;"
            "encrypt=true;trustServerCertificate=false;loginTimeout=60")
TABLES = ["inventory", "products"]


###Helper functions

In [0]:
from pyspark.sql import functions as F

# jdbc(query) sends a SQL Server query to sqldb-erp and returns the result as a Spark DataFrame. You write the SQL once and reuse the function for every query.
def jdbc(query):
    return (spark.read.format("jdbc")
            .option("url", JDBC_URL)
            .option("query", query)
            .option("user", "dbx_reader")
            .option("password", dbutils.secrets.get("kv", "sql-erp-password"))
            .option("fetchsize", "10000")   #pulls 10,000 rows per network round trip
            .load())

def rows_written(table):
    h = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()  #reads the table’s Delta transaction log.
    if h and hasattr(h, "operationMetrics") and h.operationMetrics:
        return h.operationMetrics.get("numOutputRows", "?")
    return "?"

### Read the bookmark (Databricks)

In [0]:

last = spark.sql(
    f"SELECT last_lsn FROM retail_dev.ops.cdc_watermarks WHERE source_table = 'inventory'"
).first()
print(last)   # None on the first run

###Find the end of the change log (SQL Server)

In [0]:
# Ask SQL Server for the newest LSN CDC has captured (runs inside sqldb-erp via JDBC)
max_lsn = jdbc("SELECT CONVERT(VARCHAR(22), sys.fn_cdc_get_max_lsn(), 1) AS lsn").first().lsn
print(max_lsn)


###Build the read range

In [0]:
# Lower bound: first run starts at the oldest change; later runs start just after the bookmark
if last is None:
    lo = "sys.fn_cdc_get_min_lsn('dbo_inventory')"
else:
    lo = "sys.fn_cdc_increment_lsn(CONVERT(BINARY(10), '" + last.last_lsn + "', 1))"

# Upper bound: the newest LSN from Cell 4, converted back from text to binary
hi = "CONVERT(BINARY(10), '" + max_lsn + "', 1)"

print("from:", lo)
print("to:  ", hi)

if last is not None and last.last_lsn == max_lsn:
    print("Nothing new - skip cells 6, 7 and 8")

###Preview the changes

In [0]:
changes = jdbc(
    "SELECT CONVERT(VARCHAR(22), __$start_lsn, 1) AS _lsn, "
    "CONVERT(VARCHAR(22), __$seqval, 1) AS _seqval, "
    "__$operation AS _op, * "
    "FROM cdc.fn_cdc_get_all_changes_dbo_inventory(" + lo + ", " + hi + ", 'all')"
)

print(changes.columns)
display(changes.limit(5))

###Append to Bronze (run once)

In [0]:
# Drop SQL Server's own __$ columns; their values are already in _lsn, _seqval, _op
keep = [c for c in changes.columns if not c.startswith("__$")]

(changes.select(*keep)
        .withColumn("_ingested_at", F.current_timestamp())
        .write.mode("append")
        .saveAsTable("retail_dev.bronze.erp_inventory_cdc"))

print("appended rows:", rows_written("retail_dev.bronze.erp_inventory_cdc"))

###Move the bookmark

In [0]:
spark.sql(
    "MERGE INTO retail_dev.ops.cdc_watermarks w "
    "USING (SELECT 'inventory' AS source_table, '" + max_lsn + "' AS last_lsn) s "
    "ON w.source_table = s.source_table "
    "WHEN MATCHED THEN UPDATE SET last_lsn = s.last_lsn, updated_at = now() "
    "WHEN NOT MATCHED THEN INSERT (source_table, last_lsn, updated_at) "
    "VALUES (s.source_table, s.last_lsn, now())"
)

display(spark.table("retail_dev.ops.cdc_watermarks"))

###Check Inventory in bronze

In [0]:
display(spark.sql("""
SELECT _op, count(*) AS rows, min(updated_at) AS first_change, max(updated_at) AS last_change
FROM retail_dev.bronze.erp_inventory_cdc
GROUP BY _op
ORDER BY _op
"""))

###Bookmark and newest LSN


In [0]:
last_p = spark.sql(
    "SELECT last_lsn FROM retail_dev.ops.cdc_watermarks WHERE source_table = 'products'"
).first()
print(last_p)     # None on the first run

max_lsn_p = jdbc("SELECT CONVERT(VARCHAR(22), sys.fn_cdc_get_max_lsn(), 1) AS lsn").first().lsn
print(max_lsn_p)

###Build the read range

In [0]:
if last_p is None:
    lo_p = "sys.fn_cdc_get_min_lsn('dbo_products')"
else:
    lo_p = "sys.fn_cdc_increment_lsn(CONVERT(BINARY(10), '" + last_p.last_lsn + "', 1))"

hi_p = "CONVERT(BINARY(10), '" + max_lsn_p + "', 1)"

print("from:", lo_p)
print("to:  ", hi_p)

if last_p is not None and last_p.last_lsn == max_lsn_p:
    print("Nothing new - skip cells 12, 13 and 14")

###Preview the changes

In [0]:
changes_p = jdbc(
    "SELECT CONVERT(VARCHAR(22), __$start_lsn, 1) AS _lsn, "
    "CONVERT(VARCHAR(22), __$seqval, 1) AS _seqval, "
    "__$operation AS _op, * "
    "FROM cdc.fn_cdc_get_all_changes_dbo_products(" + lo_p + ", " + hi_p + ", 'all')"
)

print(changes_p.columns)
display(changes_p.limit(5))

###Append to bronze (run once)

In [0]:
keep_p = [c for c in changes_p.columns if not c.startswith("__$")]

(changes_p.select(*keep_p)
          .withColumn("_ingested_at", F.current_timestamp())
          .write.mode("append")
          .saveAsTable("retail_dev.bronze.erp_products_cdc"))

print("appended rows:", rows_written("retail_dev.bronze.erp_products_cdc"))

###Move the bookmark

In [0]:
spark.sql(
    "MERGE INTO retail_dev.ops.cdc_watermarks w "
    "USING (SELECT 'products' AS source_table, '" + max_lsn_p + "' AS last_lsn) s "
    "ON w.source_table = s.source_table "
    "WHEN MATCHED THEN UPDATE SET last_lsn = s.last_lsn, updated_at = now() "
    "WHEN NOT MATCHED THEN INSERT (source_table, last_lsn, updated_at) "
    "VALUES (s.source_table, s.last_lsn, now())"
)

display(spark.table("retail_dev.ops.cdc_watermarks"))

###Check products in bronze

In [0]:
display(spark.sql("""
SELECT _op, count(*) AS rows, min(updated_at) AS first_change, max(updated_at) AS last_change
FROM retail_dev.bronze.erp_products_cdc
GROUP BY _op
ORDER BY _op
"""))

display(spark.sql("""
SELECT _lsn, _op, item_id, unit_price, updated_at
FROM retail_dev.bronze.erp_products_cdc
WHERE item_id = 'FOODS_3_090'
ORDER BY _lsn
"""))